# ImageNet-1K: official DeiT-III recipe + RRLSSO

This Colab entry preserves Meta's size-specific DeiT-III protocol and changes only the token mixer.

- Small: 224px / 800 epochs.
- Base and Large: 192px / 800 epochs, followed by published 224px refinement.
- Small, Base, and Large additionally support the published 384px refinement.
- Pretraining requires Apex FusedLAMB, 3-Augment, distributed-style repeated augmentation, virtual-device grouped Mixup/CutMix, BCE, FP16 AMP, EMA, official LayerScale, class-token initialization, and constant drop path.

RRLSSO uses an explicit log-gain gauge reference and a logit-space alpha saturation hinge. Both have zero initial value and gradient and are independently removable. Run cells from top to bottom.

## 1. Get the repository

In [ ]:
from pathlib import Path
import getpass, json, os, shutil, subprocess, sys, time

REPO_URL = 'https://github.com/Yang916-yy/LSSO.git'
BRANCH = 'main'
COLAB_ROOT = Path('/content') if Path('/content').is_dir() else Path.home()
ROOT = COLAB_ROOT / 'LSSO'

remote = subprocess.run(
    ['git', 'ls-remote', '--exit-code', '--heads', REPO_URL, BRANCH],
    capture_output=True, text=True,
)
assert remote.returncode == 0, remote.stderr
if not (ROOT / '.git').is_dir():
    subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch', REPO_URL, str(ROOT)], check=True)
else:
    subprocess.run(['git', '-C', str(ROOT), 'fetch', '--prune', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', str(ROOT), 'checkout', '-B', BRANCH, f'origin/{BRANCH}'], check=True)
os.chdir(ROOT)
for relative in (
    'experiments/imagenet_wds_train.py',
    'examples/models/deit3_rrlsso.py',
    'tools/hf_wds_stream.py',
):
    assert (ROOT / relative).is_file(), f'missing {relative}; push the new ImageNet code first'
print('repository:', ROOT)
print('commit:', subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], text=True).strip())

## 2. Install the matching precompiled backend

The release wheel is tied to PyTorch 2.11 and the CUDA build. This cell installs matching PyTorch only when necessary; it does not compile MathDx.

In [ ]:
nvcc = shutil.which('nvcc')
assert nvcc, 'CUDA toolkit/nvcc is unavailable'
nvcc_version = subprocess.check_output([nvcc, '--version'], text=True)
if 'release 12.8' in nvcc_version:
    CUDA_TAG = 'cu128'
    TORCH_INDEX = 'https://download.pytorch.org/whl/cu128'
    RUNTIME_WHEEL = (
        'https://github.com/Yang916-yy/LSSO/releases/download/v0.2.0/'
        'lsso_mathdx_runtime-0.2.0%2Btorch2110cu128-py3-none-linux_x86_64.whl'
    )
elif 'release 13.' in nvcc_version:
    CUDA_TAG = 'cu130'
    TORCH_INDEX = 'https://download.pytorch.org/whl/cu130'
    RUNTIME_WHEEL = (
        'https://github.com/Yang916-yy/LSSO/releases/download/v0.2.0/'
        'lsso_mathdx_runtime-0.2.0%2Btorch2110cu130-py3-none-linux_x86_64.whl'
    )
else:
    raise RuntimeError(f'Precompiled backend supports CUDA Toolkit 12.8 or 13.x, got:\n{nvcc_version}')

probe = subprocess.run(
    [sys.executable, '-c', 'import torch; print(torch.__version__, torch.version.cuda)'],
    capture_output=True, text=True,
)
expected_cuda = '12.8' if CUDA_TAG == 'cu128' else '13.0'
if probe.returncode != 0 or '2.11.0' not in probe.stdout or expected_cuda not in probe.stdout:
    assert 'torch' not in sys.modules, 'Restart the runtime, then run from cell 1'
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q', '--upgrade',
         'torch==2.11.0', 'torchvision==0.26.0', '--index-url', TORCH_INDEX],
        check=True,
    )
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[experiments]'], cwd=ROOT, check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', RUNTIME_WHEEL], check=True)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ninja', 'packaging'], check=True)
apex_probe = subprocess.run(
    [sys.executable, '-c', 'from apex.optimizers import FusedLAMB; print(FusedLAMB)'],
    capture_output=True, text=True,
)
if apex_probe.returncode != 0:
    APEX_ROOT = COLAB_ROOT / 'apex'
    if not (APEX_ROOT / '.git').is_dir():
        subprocess.run(['git', 'clone', 'https://github.com/NVIDIA/apex.git', str(APEX_ROOT)], check=True)
    subprocess.run(['git', '-C', str(APEX_ROOT), 'fetch', 'origin', '6424da3b4faa6c8f062da4a48c424fff3f02d42d'], check=True)
    subprocess.run(['git', '-C', str(APEX_ROOT), 'checkout', '--detach', '6424da3b4faa6c8f062da4a48c424fff3f02d42d'], check=True)
    apex_env = os.environ.copy()
    apex_env.update({
        'APEX_CPP_EXT': '1',
        'APEX_CUDA_EXT': '1',
        'APEX_DEPRECATED_FUSED_LAMB': '1',
        'APEX_PARALLEL_BUILD': str(min(16, os.cpu_count() or 4)),
        'NVCC_APPEND_FLAGS': '--threads 4',
    })
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-v', '--no-build-isolation', '.'],
        cwd=APEX_ROOT, env=apex_env, check=True,
    )
subprocess.run(
    [sys.executable, '-c', 'from apex.optimizers import FusedLAMB; print("Apex FusedLAMB loaded")'],
    check=True,
)

print(subprocess.check_output(
    [sys.executable, '-c', 'import torch; print("torch", torch.__version__, "CUDA", torch.version.cuda)'],
    text=True,
).strip())

## 3. Authenticate and configure official Base pretraining

Directories stay shallow. The Base physical batch is 512 with four-step accumulation, reproducing effective batch 2048. Each forward contains two independent 256-sample virtual augmentation groups, so every optimizer update retains the official eight group-level Mixup/CutMix draws.

In [ ]:
hf_token = os.environ.get('HF_TOKEN')
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN') or hf_token
except Exception:
    pass
if not hf_token:
    hf_token = getpass.getpass('Hugging Face token: ')
assert hf_token.startswith('hf_'), 'Expected a Hugging Face access token'
os.environ['HF_TOKEN'] = hf_token
del hf_token

content_root = Path('/content') if Path('/content').is_dir() else Path.home()
cache_parent = content_root
if shutil.disk_usage(cache_parent).free / 2**30 < 155:
    local_scratch = Path('/mnt/local-scratch')
    if local_scratch.is_dir() and os.access(local_scratch, os.W_OK):
        cache_parent = local_scratch
CACHE_ROOT = cache_parent / 'imagenet-wds'
CHECKPOINT_ROOT = content_root / 'checkpoints'
ARCHIVE_ROOT = content_root / 'archives'
for directory in (CACHE_ROOT, CHECKPOINT_ROOT, ARCHIVE_ROOT):
    directory.mkdir(parents=True, exist_ok=True)

CONFIG = {
    'model': 'deit3_base_patch16_rrlsso',
    'stage': 'pretrain',
    'image_size': 192,
    'epochs': 800,
    'rank': 32,
    'batch_size': 512,
    'eval_batch_size': 512,
    'grad_accum': 4,
    'workers': 32,
    'eval_workers': 4,
    'seed': 0,
    'extended_diagnostics': False,
    'cache_dir': str(CACHE_ROOT),
    'output': str(CHECKPOINT_ROOT / 'deit3_base_rrlsso_pre192'),
    'init_checkpoint': '',
}
Path(CONFIG['output']).mkdir(parents=True, exist_ok=True)
print(json.dumps(CONFIG, indent=2))
print('cache filesystem:', CACHE_ROOT)

## 4. Check registration, GPU, disk, and native backend

In [ ]:
import gc
import torch, timm
import examples.models
from experiments.imagenet_wds_train import parse_args
from lsso.mathdx_backend import is_mathdx_available, mathdx_load_error

assert torch.cuda.is_available(), 'CUDA is unavailable'
assert timm.is_model(CONFIG['model']), f"model is not registered: {CONFIG['model']}"
assert is_mathdx_available(), f'precompiled backend failed to load: {mathdx_load_error()}'

free_gib = shutil.disk_usage(CACHE_ROOT).free / 2**30
cached_gib = sum(p.stat().st_size for p in CACHE_ROOT.glob('*.tar')) / 2**30
assert free_gib + cached_gib >= 150, (
    f'Need roughly 150 GiB for the complete cache; free={free_gib:.1f}, cached={cached_gib:.1f} GiB'
)
gpu_name = torch.cuda.get_device_name()
gpu_gib = torch.cuda.get_device_properties(0).total_memory / 2**30
model = timm.create_model(CONFIG['model'], img_size=32, num_classes=1000, rank=CONFIG['rank'])
parameters = sum(p.numel() for p in model.parameters())
recipe_args = parse_args([
    '--model', CONFIG['model'], '--stage', CONFIG['stage'],
    '--batch-size', str(CONFIG['batch_size']), '--grad-accum', str(CONFIG['grad_accum']),
])
assert recipe_args.optimizer == 'fusedlamb'
assert recipe_args.augmentation == 'three_augment'
assert recipe_args.bce_loss and recipe_args.repeated_aug == 3
assert recipe_args.drop_path_rate == 0.2
assert recipe_args.rrlsso_gain_reg == recipe_args.rrlsso_alpha_reg == 1e-4
assert recipe_args.rrlsso_alpha_saturation == 0.8
assert CONFIG['batch_size'] * CONFIG['grad_accum'] == recipe_args.effective_batch == 2048
assert recipe_args.augmentation_group_size == recipe_args.runtime_augmentation_group_size == 256
assert CONFIG['batch_size'] % recipe_args.augmentation_group_size == 0
del model
gc.collect()
print(f'GPU: {gpu_name} ({gpu_gib:.1f} GiB)')
print(f'model: {CONFIG["model"]} | parameters: {parameters:,} | rank: {CONFIG["rank"]}')
print(f'disk: {free_gib:.1f} GiB free + {cached_gib:.1f} GiB cached')
print('backend: precompiled MathDx/CUDA ABI 1 loaded')
print('recipe: official Base 192/800 | FusedLAMB | virtual groups 256 | distributed RAx3 | BCE | FP16 | EMA')
print('transport: one HTTP request + one locked cache owner per shard; no prefetch manager')

## 5. Final target-GPU gate

Run this once on the actual cloud GPU before starting the detached job. Unlike
the old batch-8 smoke, this gate uses the configured physical batch, gradient
accumulation, train workers, validation workers, native backend, and optimizer.
It performs one complete optimizer update and validation, saves `last.pt`, then
resumes for one more update. It rejects missing EMA/optimizer/scheduler state or
a moved gain reference. One train shard and one validation shard bound the data
download. The temporary gate directory is deleted only after every check passes.

In [ ]:
def train_command(config, *, output=None, resume=True, epochs=None):
    command = [
        sys.executable, '-u', 'experiments/imagenet_wds_train.py',
        '--model', config['model'],
        '--stage', config['stage'],
        '--epochs', str(config['epochs'] if epochs is None else epochs),
        '--rank', str(config['rank']),
        '--cache-dir', config['cache_dir'],
        '--output', str(output or config['output']),
        '--batch-size', str(config['batch_size']),
        '--eval-batch-size', str(config['eval_batch_size']),
        '--grad-accum', str(config['grad_accum']),
        '--workers', str(config['workers']),
        '--eval-workers', str(config['eval_workers']),
        '--seed', str(config['seed']),
        '--require-mathdx',
        '--resume' if resume else '--no-resume',
    ]
    if config.get('extended_diagnostics'):
        command.append('--rrlsso-extended-diagnostics')
    # Initialization and resume are deliberately disjoint checkpoint states.
    if not resume and config.get('init_checkpoint'):
        command += ['--init-checkpoint', config['init_checkpoint']]
    return command

import csv
import hashlib

gate_output = CHECKPOINT_ROOT / f'_gate_{CONFIG["model"]}_{CONFIG["stage"]}_{int(time.time())}'
gate_steps = max(1, int(CONFIG['grad_accum']))  # exactly one optimizer update

def gain_reference_digest(checkpoint):
    reference = checkpoint['rrlsso_gain_reference']
    digest = hashlib.sha256()
    for name, value in sorted(reference.items()):
        digest.update(name.encode())
        digest.update(value.detach().cpu().contiguous().numpy().tobytes())
    return digest.hexdigest()

def load_gate_checkpoint():
    checkpoint_path = gate_output / 'last.pt'
    assert checkpoint_path.is_file(), f'missing {checkpoint_path}'
    checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
    for key in ('model', 'model_ema', 'optimizer', 'scheduler', 'scaler',
                'rrlsso_gain_reference', 'run_metadata', 'rng_state'):
        assert checkpoint.get(key) is not None, f'checkpoint is missing {key}'
    assert checkpoint['run_metadata']['training_stage'] == CONFIG['stage']
    assert checkpoint['run_metadata']['rank'] == CONFIG['rank']
    return checkpoint

gate_common = [
    '--steps-per-epoch', str(gate_steps),
    '--max-val-steps', '1',
    '--shard-limit', '1',
    '--log-interval', '1',
]
started = time.time()
print(f'Gate phase 1/2: formal batch={CONFIG["batch_size"]}, '
      f'accumulation={CONFIG["grad_accum"]}, workers={CONFIG["workers"]}')
subprocess.run(
    train_command(CONFIG, output=gate_output, resume=False, epochs=1)
    + gate_common + ['--overwrite-output'],
    cwd=ROOT, env=os.environ.copy(), check=True,
)
first = load_gate_checkpoint()
assert first['epoch'] == 1 and first['global_update'] == 1
first_reference = gain_reference_digest(first)

gate_config = json.loads((gate_output / 'config.json').read_text())
expected_optimizer = 'apex_fusedlamb' if CONFIG['stage'] == 'pretrain' else 'adamw'
assert gate_config['optimizer_impl'] == expected_optimizer, gate_config['optimizer_impl']
assert gate_config['actual_effective_batch'] == CONFIG['batch_size'] * CONFIG['grad_accum']

print('Gate phase 2/2: resume without reinitializing the stage')
subprocess.run(
    train_command(CONFIG, output=gate_output, resume=True, epochs=2) + gate_common,
    cwd=ROOT, env=os.environ.copy(), check=True,
)
second = load_gate_checkpoint()
assert second['epoch'] == 2 and second['global_update'] == 2
assert gain_reference_digest(second) == first_reference, 'gain reference moved on resume'

with (gate_output / 'metrics.csv').open(newline='') as handle:
    rows = list(csv.DictReader(handle))
assert len(rows) == 2
peak_gib = max(float(row['peak_gb']) for row in rows)
assert 0.0 < peak_gib < gpu_gib
elapsed = time.time() - started
shutil.rmtree(gate_output)
print(f'FINAL GATE PASSED in {elapsed / 60:.1f} min | peak={peak_gib:.2f} GiB')
print('Checkpoint resume, gain reference, native backend requirement, formal optimizer,')
print('physical batch, accumulation, DataLoader pools, validation, and disk writes are healthy.')

## 6. Launch or resume detached training

Run once. The cell chooses exactly one state: a new pretraining run, a new
refinement initialized from `init_checkpoint`, or a resume from this stage's
own `last.pt`. A refinement resume never sends the initialization checkpoint,
so its saved gain reference cannot move. Existing `last.pt` restores model,
EMA, optimizer, epoch, global update, LR progress, gain reference, and saved RNG
states. The exact WebDataset cursor remains newly stochastic after restart.

In [ ]:
output_dir = Path(CONFIG['output'])
output_dir.mkdir(parents=True, exist_ok=True)
pid_path = output_dir / 'trainer.pid'
log_path = output_dir / 'train.log'

def trainer_is_live(pid):
    try:
        state = Path(f'/proc/{pid}/stat').read_text().split()[2]
        cmdline = Path(f'/proc/{pid}/cmdline').read_bytes().replace(b'\0', b' ').decode(errors='replace')
        return state != 'Z' and 'experiments/imagenet_wds_train.py' in cmdline and str(output_dir) in cmdline
    except (FileNotFoundError, PermissionError, ProcessLookupError, ValueError, IndexError):
        return False

if pid_path.is_file():
    old_pid = int(pid_path.read_text().strip())
    if trainer_is_live(old_pid):
        raise RuntimeError(f'trainer PID {old_pid} is already running')
    pid_path.unlink(missing_ok=True)

has_last = (output_dir / 'last.pt').is_file()
if CONFIG['stage'] == 'pretrain':
    launch_resume = True  # auto-resume when last.pt exists; otherwise start new pretraining
else:
    launch_resume = has_last
    if not launch_resume:
        init_checkpoint = Path(CONFIG.get('init_checkpoint', ''))
        assert init_checkpoint.is_file(), f'missing refinement initializer: {init_checkpoint}'
launch_mode = ('resume_' + CONFIG['stage']) if has_last else (
    'new_pretrain' if CONFIG['stage'] == 'pretrain' else 'init_' + CONFIG['stage']
)
print('checkpoint mode:', launch_mode)

log = log_path.open('a', buffering=1)
process = subprocess.Popen(
    train_command(CONFIG, resume=launch_resume),
    cwd=ROOT, env=os.environ.copy(), stdout=log, stderr=subprocess.STDOUT,
    start_new_session=True, close_fds=True,
)
log.close()
pid_path.write_text(str(process.pid))
time.sleep(3)
if process.poll() is not None:
    print('\n'.join(log_path.read_text(errors='replace').splitlines()[-50:]))
    raise RuntimeError(f'trainer exited during startup: {process.returncode}')
print('trainer started:', process.pid)
print('output:', output_dir)
print('log:', log_path)
print('Interrupting the independent monitor below does not stop training.')

## 7. Independent 120-second monitor

In [ ]:
from IPython.display import clear_output

monitor_output = Path(CONFIG['output'])
monitor_cache = Path(CONFIG['cache_dir'])

def find_trainers():
    matches = []
    for proc_dir in Path('/proc').glob('[0-9]*'):
        try:
            cmdline = proc_dir.joinpath('cmdline').read_bytes().replace(b'\0', b' ').decode(errors='replace')
            stat = proc_dir.joinpath('stat').read_text().split()
            pid, parent, state = int(proc_dir.name), int(stat[3]), stat[2]
        except (FileNotFoundError, PermissionError, ProcessLookupError, ValueError, IndexError):
            continue
        if 'experiments/imagenet_wds_train.py' in cmdline and str(monitor_output) in cmdline and state != 'Z':
            matches.append((pid, parent, state))
    pids = {pid for pid, _, _ in matches}
    roots = [(pid, state) for pid, parent, state in matches if parent not in pids]
    return roots, len(matches) - len(roots)

def tail(path, lines):
    path = Path(path)
    if not path.is_file():
        return '(not created yet)'
    content = path.read_text(errors='replace').splitlines()
    return '\n'.join(content[-lines:]) if content else '(empty)'

try:
    while True:
        clear_output(wait=True)
        print(time.strftime('%Y-%m-%d %H:%M:%S'))
        trainers, loader_workers = find_trainers()
        print('trainer:', trainers if trainers else 'not found')
        print('DataLoader workers:', loader_workers)
        subprocess.run([
            'nvidia-smi',
            '--query-gpu=name,utilization.gpu,utilization.memory,memory.used,memory.total,temperature.gpu,power.draw',
            '--format=csv,noheader',
        ], check=False)
        shards = list(monitor_cache.glob('*.tar'))
        partials = list(monitor_cache.glob('*.partial'))
        shard_gib = sum(p.stat().st_size for p in shards) / 2**30
        partial_gib = sum(p.stat().st_size for p in partials) / 2**30
        print(f'HF cache: {len(shards)} complete ({shard_gib:.2f} GiB), '
              f'{len(partials)} active ({partial_gib:.2f} GiB)')
        print('--- metrics ---')
        print(tail(monitor_output / 'metrics.csv', 6))
        print('--- log tail ---')
        print(tail(monitor_output / 'train.log', 20))
        last = monitor_output / 'last.pt'
        if last.is_file():
            updated = time.strftime('%Y-%m-%d %H:%M:%S', time.localtime(last.stat().st_mtime))
            print(f'last.pt: {last.stat().st_size / 2**30:.2f} GiB, updated {updated}')
        else:
            print('last.pt: not created yet (saved after the first complete epoch)')
        print('next refresh in 120 seconds')
        for _ in range(24):
            time.sleep(5)
except KeyboardInterrupt:
    print('Monitor stopped; trainer was not signaled.')

## 8. Create and download a resume archive

Stop only the monitor cell first. Training may continue while this takes a stable copy of last.pt. best.pt is excluded by default to keep the archive small.

In [ ]:
from datetime import datetime
import gc, zipfile
from google.colab import files

run_dir = Path(CONFIG['output'])
last_source = run_dir / 'last.pt'
assert last_source.is_file(), 'finish at least one epoch first'
INCLUDE_BEST = False
stamp = datetime.now().strftime('%Y%m%d-%H%M%S')
stage_dir = ARCHIVE_ROOT / f'{run_dir.name}-{stamp}'
archive_path = ARCHIVE_ROOT / f'{run_dir.name}-{stamp}.zip'
stage_dir.mkdir(parents=True, exist_ok=False)

def stable_copy(source, destination, attempts=10):
    for _ in range(attempts):
        before = source.stat()
        signature = before.st_size, before.st_mtime_ns
        shutil.copy2(source, destination)
        after = source.stat()
        if signature == (after.st_size, after.st_mtime_ns) and destination.stat().st_size == after.st_size:
            return
        destination.unlink(missing_ok=True)
        time.sleep(3)
    raise RuntimeError(f'{source.name} kept changing; retry between checkpoint writes')

try:
    stable_copy(last_source, stage_dir / 'last.pt')
    if INCLUDE_BEST and (run_dir / 'best.pt').is_file():
        stable_copy(run_dir / 'best.pt', stage_dir / 'best.pt')
    for name in ('config.json', 'metrics.csv', 'train.log'):
        if (run_dir / name).is_file():
            shutil.copy2(run_dir / name, stage_dir / name)
    checkpoint = torch.load(stage_dir / 'last.pt', map_location='cpu', weights_only=False, mmap=True)
    assert {'model', 'optimizer', 'epoch', 'global_update'} <= checkpoint.keys()
    print('validated:', {key: checkpoint.get(key) for key in ('epoch', 'global_update', 'best_acc')})
    del checkpoint
    gc.collect()
    with zipfile.ZipFile(archive_path, 'w', compression=zipfile.ZIP_STORED, allowZip64=True) as bundle:
        for source in sorted(stage_dir.iterdir()):
            bundle.write(source, arcname=f'{run_dir.name}/{source.name}')
    with zipfile.ZipFile(archive_path) as bundle:
        assert bundle.testzip() is None
finally:
    shutil.rmtree(stage_dir, ignore_errors=True)
print(f'archive: {archive_path} ({archive_path.stat().st_size / 2**30:.2f} GiB)')
files.download(str(archive_path))

## 9. Switch to the official 224px Base refinement

Run only after Base 192px pretraining finishes. Then rerun the smoke and launch
sections. On the first launch, the raw `best.pt` model initializes refinement
and its loaded gain values become the fixed reference. On later launches, this
stage's `last.pt` is resumed and the original reference is restored unchanged;
the pretraining checkpoint is not loaded again.

In [ ]:
pretrain_dir = CHECKPOINT_ROOT / 'deit3_base_rrlsso_pre192'
pretrain_best = pretrain_dir / 'best.pt'
assert pretrain_best.is_file(), f'missing {pretrain_best}'
CONFIG.update({
    'stage': 'finetune224',
    'image_size': 224,
    'epochs': 20,
    'batch_size': 512,
    'eval_batch_size': 512,
    'grad_accum': 1,
    'output': str(CHECKPOINT_ROOT / 'deit3_base_rrlsso_ft224'),
    'init_checkpoint': str(pretrain_best),
})
Path(CONFIG['output']).mkdir(parents=True, exist_ok=True)
print(json.dumps(CONFIG, indent=2))
print('Rerun cells 5–8. Learned PE is interpolated from 12x12 to 14x14.')

## Notes

- Registered names are resolution-independent: deit3_small_patch16_rrlsso, deit3_base_patch16_rrlsso, and deit3_large_patch16_rrlsso.
- Formal pretraining and the final cloud gate refuse to start without Apex FusedLAMB.
- Large physical batches are divided into official per-GPU virtual groups before Mixup/CutMix; RAx3 views are separated across groups.
- The 1e-4 log-gain gauge anchor and logit-space alpha hinge have exactly zero initial loss and gradient. Refinement anchors gain to the loaded checkpoint rather than to one. Set both weights to zero for the exact-recipe ablation.
- By default, metrics.csv keeps only gain drift, alpha distributions/saturation, barrier magnitude, and update count. Set CONFIG['extended_diagnostics']=True only for pilot/debug runs that need gradient ratios and actual solve-scalar update norms.
- For 384px refinement, use stage finetune384 with effective batch 512 and the official virtual group size selected by the trainer.
- The final gate uses the configured full physical batch and can take several minutes; it replaces the old batch-8 smoke.
- Never launch two sizes simultaneously. Stopping the monitor does not signal the detached trainer.
- The simple one-request/one-cache-owner Hugging Face stream remains unchanged.